In [1]:
import torch, torch.optim as optim
import numpy as np
import gc
import warnings
import torch.nn.functional as F
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from IPython.utils import io
from reader import prepare_qsm_dataset
from loader import get_slice_level_data
from util import seed_everything, compute_comprehensive_metrics, get_cv_fold_metrics, fmt_cv
from util import mask_crop as mask_crop_fn
from train import calibrate_balanced
from validate import ClinicalTransformer_cv, ResidualSpectralViT_cv
from networks import FocalLoss, PassThrough, ClinicalTransformer, SpectralViT, SpatialViT, ResidualSpectral, ResidualSpatial, AttentionUNet

warnings.filterwarnings("ignore", category=UserWarning, message="Default upsampling behavior")

# Configuration
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
N_FOLDS, EPOCHS, JITTER_STD, IMG_AUG_STD, TARGET_DIM, EMBED_DIM = 5, 100, 0.1, 0.05, 128, 32
seed_everything(0)

# Load datasets
dataset_train = prepare_qsm_dataset('MSW', '/media/mts_dbs/dbs/all/nii/qsm_115/im', '/media/mts_dbs/dbs/all/nii/seg_ps/', '/data/Ali/RadDBS-QSM/data/docs/dbs_03292024.csv', './msw_cache_6d_cv.pt', load_cache=True, mask_crop_fn=mask_crop_fn, cv_pad=False)
dataset_test = prepare_qsm_dataset('CHH', '/media/mts_dbs/chh/nii/qsm/', '/media/mts_dbs/chh/roi/', '/media/mts_dbs/chh/xlsx/chh_subjects_table1_20240729.csv', './chh_cache_6d_cv.pt', load_cache=True, mask_crop_fn=mask_crop_fn, cv_pad=False)

# Extract Arrays
X_full_slices, X_full_clin, y_full_slices, full_subj_map = get_slice_level_data(dataset_train, TARGET_DIM, include_unlabeled=True)
labeled_mask = (y_full_slices != -1)
X_tr_slices, X_tr_clin_raw, y_tr_slices, tr_subj_map = X_full_slices[labeled_mask], X_full_clin[labeled_mask], y_full_slices[labeled_mask], full_subj_map[labeled_mask]
X_te_slices, X_te_clin_raw, y_te_slices, te_subj_map = get_slice_level_data(dataset_test, TARGET_DIM)

# Harmonization & Weights
scaler_train = StandardScaler()
X_tr_clin = scaler_train.fit_transform(X_tr_clin_raw)
scaler_test = StandardScaler()
X_te_clin = scaler_test.fit(X_te_clin_raw[((X_te_clin_raw.shape[0])//4):,:]).transform(X_te_clin_raw)
NEG_WEIGHT = float(sum(y_tr_slices == 1) // sum(y_tr_slices == 0))
GAMMA = NEG_WEIGHT

# Hyperparameter selection
unique_subjs = np.unique(tr_subj_map)
y_unique = np.array([y_tr_slices[tr_subj_map == s][0] for s in unique_subjs])
outer_skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
criterion = FocalLoss(gamma=GAMMA)

SELECTED_POS_WEIGHT = ClinicalTransformer_cv(model_class=ClinicalTransformer, model_kwargs={'n_inputs': X_tr_clin.shape[1]}, pos_weight_grid=[0.1, 0.25, 0.5, 0.75, 1.0], outer_skf=outer_skf, unique_subjs=unique_subjs, y_unique=y_unique, tr_subj_map=tr_subj_map, X_tr_clin=X_tr_clin, y_tr_slices=y_tr_slices, NEG_WEIGHT=NEG_WEIGHT, criterion=criterion, JITTER_STD=JITTER_STD, EPOCHS=EPOCHS, device=device)

model_classes = {'clinical': ClinicalTransformer, 'vit': SpectralViT, 'wrapper': ResidualSpectral}
model_kwargs = {'vit': {'n_heads': 1, 'n_layers': 1, 'embed_dim': 32, 'use_input_proj': False, 'use_pos_embed': False, 'use_layer_norm': False, 'pooling': 'flatten'}}

N_PCA_COMPONENTS = ResidualSpectralViT_cv(model_classes=model_classes, model_kwargs=model_kwargs, pca_components_grid=[16, 32, 64, 128], outer_skf=outer_skf, unique_subjs=unique_subjs, y_unique=y_unique, tr_subj_map=tr_subj_map, X_tr_slices=X_tr_slices, X_tr_clin=X_tr_clin, y_tr_slices=y_tr_slices, NEG_WEIGHT=NEG_WEIGHT, SELECTED_POS_WEIGHT=SELECTED_POS_WEIGHT, criterion=criterion, JITTER_STD=JITTER_STD, EPOCHS=EPOCHS, device=device)

# Cross-validation
print("Training...")
clinical_fold_weights = []
eval_skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
all_cv_probs_ct, all_cv_probs_sp, all_cv_probs_va, all_cv_labels = [], [], [], []
fold_preds_ct, fold_preds_sp, fold_preds_spatial = [], [], []
fold_thresholds_ct, fold_thresholds_sp, fold_thresholds_va = [], [], []

for fold_idx, (train_subj_idx, val_subj_idx) in enumerate(eval_skf.split(unique_subjs, y_unique)):
    print(f"Fold {fold_idx + 1}/{N_FOLDS}")
    train_mask, val_mask = np.isin(tr_subj_map, unique_subjs[train_subj_idx]), np.isin(tr_subj_map, unique_subjs[val_subj_idx])
    
    f_img_scaler = StandardScaler().fit(X_tr_slices[train_mask])
    f_pca = PCA(n_components=N_PCA_COMPONENTS, random_state=0, whiten=True).fit(f_img_scaler.transform(X_tr_slices[train_mask]))
    
    X_train_pca, X_train_clin, X_train_img, y_train = f_pca.transform(f_img_scaler.transform(X_tr_slices[train_mask])), X_tr_clin[train_mask], X_tr_slices[train_mask], y_tr_slices[train_mask]
    X_val_pca, X_val_clin, X_val_img, y_val = f_pca.transform(f_img_scaler.transform(X_tr_slices[val_mask])), X_tr_clin[val_mask], X_tr_slices[val_mask], y_tr_slices[val_mask]
    X_test_pca, X_test_clin, X_test_img = f_pca.transform(f_img_scaler.transform(X_te_slices)), X_te_clin, X_te_slices
    
    X_train_clin_t, X_train_pca_t = torch.tensor(X_train_clin, dtype=torch.float32).to(device), torch.tensor(X_train_pca, dtype=torch.float32).to(device)
    X_train_img_t, y_train_t = torch.tensor(X_train_img).view(-1, 1, 128, 128).float().to(device), torch.tensor(y_train, dtype=torch.float32).to(device)
    w = torch.where(y_train_t == 0, torch.tensor(NEG_WEIGHT, device=device), torch.tensor(SELECTED_POS_WEIGHT, device=device))

    m_ct = ClinicalTransformer(n_inputs=X_tr_clin.shape[1]).to(device)
    opt_ct = optim.Adam(m_ct.parameters(), lr=1e-4)
    for _ in range(EPOCHS):
        m_ct.train(); opt_ct.zero_grad()
        loss = criterion(m_ct(X_train_clin_t + torch.randn_like(X_train_clin_t) * JITTER_STD), y_train_t, weight=w)
        loss.backward(); opt_ct.step()
    
    m_ct.eval(); [p.requires_grad_(False) for p in m_ct.parameters()]
    clinical_fold_weights.append(m_ct.state_dict())

    m_sp = ResidualSpectral(m_ct, SpectralViT(n_inputs=N_PCA_COMPONENTS, n_heads=1, n_layers=1, embed_dim=EMBED_DIM, use_input_proj=False, use_pos_embed=False, use_layer_norm=False, pooling='flatten')).to(device)
    m_va = ResidualSpatial(m_ct, SpatialViT(size=128, patch_size=16, embed_dim=EMBED_DIM, n_heads=1, n_layers=1, dropout=0.1, is_2d=True, use_cls_token=False, use_layer_norm=False)).to(device)
    
    opt_va = optim.Adam(m_va.m_res.parameters(), lr=1e-4)
    for _ in range(EPOCHS):
        m_va.train(); opt_va.zero_grad()
        loss = criterion(m_va(X_train_img_t + torch.randn_like(X_train_img_t) * IMG_AUG_STD, X_train_clin_t), y_train_t, weight=w)
        loss.backward(); opt_va.step()

    with torch.no_grad():
        v_probs_ct, v_probs_sp, v_probs_va, v_labels = [], [], [], []
        for s in unique_subjs[val_subj_idx]:
            m = (tr_subj_map[val_mask] == s); v_labels.append(y_val[m][0])
            v_probs_ct.append(m_ct(torch.tensor(X_val_clin[m], dtype=torch.float32).to(device)).mean().item())
            v_probs_sp.append(m_sp(torch.tensor(X_val_pca[m], dtype=torch.float32).to(device), torch.tensor(X_val_clin[m], dtype=torch.float32).to(device)).mean().item())
            v_probs_va.append(m_va(torch.tensor(X_val_img[m]).view(-1, 1, 128, 128).float().to(device), torch.tensor(X_val_clin[m], dtype=torch.float32).to(device)).mean().item())

        fold_thresholds_ct.append(calibrate_balanced(PassThrough(), None, np.array(v_probs_ct), np.array(v_labels), 'cpu'))
        fold_thresholds_sp.append(calibrate_balanced(PassThrough(), None, np.array(v_probs_sp), np.array(v_labels), 'cpu'))
        fold_thresholds_va.append(calibrate_balanced(PassThrough(), None, np.array(v_probs_va), np.array(v_labels), 'cpu'))
        all_cv_probs_ct.extend(v_probs_ct); all_cv_probs_sp.extend(v_probs_sp); all_cv_probs_va.extend(v_probs_va); all_cv_labels.extend(v_labels)

        t_probs_ct, t_probs_sp, t_probs_va = [], [], []
        for s in np.unique(te_subj_map):
            m = (te_subj_map == s)
            t_probs_ct.append(m_ct(torch.tensor(X_test_clin[m], dtype=torch.float32).to(device)).mean().item())
            t_probs_sp.append(m_sp(torch.tensor(X_test_pca[m], dtype=torch.float32).to(device), torch.tensor(X_test_clin[m], dtype=torch.float32).to(device)).mean().item())
            t_probs_va.append(m_va(torch.tensor(X_test_img[m]).view(-1, 1, 128, 128).float().to(device), torch.tensor(X_test_clin[m], dtype=torch.float32).to(device)).mean().item())
        fold_preds_ct.append(t_probs_ct); fold_preds_sp.append(t_probs_sp); fold_preds_spatial.append(t_probs_va)

th_ct, th_sp, th_va = np.mean(fold_thresholds_ct), np.mean(fold_thresholds_sp), np.mean(fold_thresholds_va)
y_test_labels = np.array([y_te_slices[te_subj_map == s][0] for s in np.unique(te_subj_map)])
cv_probs = {"Clinical": all_cv_probs_ct, "Spectral": all_cv_probs_sp, "Spatial": all_cv_probs_va}
test_probs = {"Clinical": np.mean(fold_preds_ct, axis=0), "Spectral": np.mean(fold_preds_sp, axis=0), "Spatial": np.mean(fold_preds_spatial, axis=0)}
thresholds = {"Clinical": th_ct, "Spectral": th_sp, "Spatial": th_va}


Preparing MSW dataset (Forced 6-dim alignment) 
Pre-flight check: Validating MSW CSV mapping...
--- MSW CSV RAW MEANS ---
  > Age     : 62.13
  > Sex     : 0.26
  > Dur     : 8.47
  > LEDD    : 989.80
  > Off-Pre : 45.91
  > On-Pre  : 19.60

Final MSW Breakdown:
 - Unique Subjects on Disk: 111
 - Labeled Responders (1): 61
 - Labeled Non-Responders (0): 5
 - Unlabeled subjects (-1): 45
 - Verified Realized Means (Matched Data Only):
    > Age     : 63.08
    > Sex     : 0.26
    > Dur     : 8.44
    > LEDD    : 1003.95
    > Off-Pre : 45.62
    > On-Pre  : 19.55

❌ FULL MISSING LIST (45 subjects):
  [3, 4, 5, 8, 12, 13, 14, 17, 18, 21, 22, 24, 25, 27, 28, 31, 32, 34, 35, 37, 39, 40, 41, 42, 49, 50, 52, 54, 57, 61, 65, 67, 74, 76, 81, 82, 84, 88, 89, 94, 99, 101, 104, 105, 116]
Loaded cache with 7790 slices.

Preparing CHH dataset (Forced 6-dim alignment) 
Pre-flight check: Validating CHH CSV mapping...
--- CHH CSV RAW MEANS ---
  > Age     : 63.13
  > Sex     : 0.46
  > Dur     : 8.54

In [2]:
# Flush and initialize
all_cv_probs_un, fold_preds_un, fold_ths_un, scaler_amp = [], [], [], GradScaler()

for fold, (t_subj_idx, v_subj_idx) in enumerate(eval_skf.split(unique_subjs, y_unique)):
    print(f"Fold {fold+1} Attention U-Net: ", end='', flush=True)
    
    # Setup data & clinical model
    t_mask = np.isin(tr_subj_map, unique_subjs[t_subj_idx])
    scaler_c = StandardScaler().fit(X_tr_clin[t_mask])
    m_ct_fold = ClinicalTransformer(n_inputs=X_tr_clin.shape[1]).to(device)
    m_ct_fold.load_state_dict(clinical_fold_weights[fold]); m_ct_fold.eval()
    
    with torch.no_grad():
        xt_c_gpu = torch.tensor(scaler_c.transform(X_tr_clin[t_mask]), dtype=torch.float32).to(device)
        t_clin_logits = m_ct_fold(xt_c_gpu, return_logit=True).detach()

    xt_i_gpu = torch.tensor(X_tr_slices[t_mask]).view(-1, 1, 128, 128).float().to(device)
    yt_gpu = torch.tensor(y_tr_slices[t_mask], dtype=torch.float32).to(device)
    pos_idx, neg_idx = torch.where(yt_gpu == 1)[0], torch.where(yt_gpu == 0)[0]
    
    batch_size, half_batch = 64, 32
    num_batches = len(pos_idx) // half_batch

    # Replicate FastUNet
    m_un_res = AttentionUNet(
        in_channels=1, 
        base_channels=16,
        fast_mode=True   
    ).to(device)
    optimizer = optim.Adam(m_un_res.parameters(), lr=5e-4)
   
    # Training Loop
    for epoch in range(EPOCHS):
        m_un_res.train()
        shuffled_neg = neg_idx[torch.randperm(len(neg_idx))]
        
        for i in range(num_batches):
            n_ids = shuffled_neg[i*half_batch : (i+1)*half_batch]
            p_ids = pos_idx[torch.randint(0, len(pos_idx), (half_batch,))]
            b_idx = torch.cat([p_ids, n_ids])
            
            optimizer.zero_grad(set_to_none=True) 
            with autocast():
                # Correct logit summation
                res_logit = m_un_res(xt_i_gpu[b_idx] + torch.randn_like(xt_i_gpu[b_idx]) * IMG_AUG_STD)
                joint_logit = res_logit + t_clin_logits[b_idx]
                
                bce_loss = F.binary_cross_entropy_with_logits(joint_logit, yt_gpu[b_idx], reduction='none')
                w_batch = torch.where(yt_gpu[b_idx] == 0, torch.tensor(NEG_WEIGHT, device=device), torch.tensor(SELECTED_POS_WEIGHT, device=device))
                loss = (w_batch * (1 - torch.exp(-bce_loss))**GAMMA * bce_loss).mean()
            
            scaler_amp.scale(loss).backward()
            scaler_amp.step(optimizer)
            scaler_amp.update()
            
        if (epoch + 1) % 25 == 0: print(f'{epoch+1}..', end='', flush=True)
    print('Done.')

    # 4. Inference
    m_un_res.eval()
    with torch.no_grad():
        v_un, v_lbls = [], []
        for s_id in unique_subjs[v_subj_idx]:
            sm = (tr_subj_map == s_id)
            c_v = torch.tensor(scaler_c.transform(X_tr_clin[sm]), dtype=torch.float32).to(device)
            i_v = torch.tensor(X_tr_slices[sm]).view(-1, 1, 128, 128).float().to(device)
            l_c = m_ct_fold(c_v, return_logit=True)
            v_un.append(torch.sigmoid(l_c + m_un_res(i_v)).mean().item())
            v_lbls.append(y_tr_slices[sm][0])
            
        fold_ths_un.append(calibrate_balanced(PassThrough(), None, np.array(v_un), np.array(v_lbls), 'cpu'))
        all_cv_probs_un.extend(v_un)

        t_un = []
        for s_id in np.unique(te_subj_map):
            sm = (te_subj_map == s_id)
            c_t = torch.tensor(scaler_c.transform(X_te_clin[sm]), dtype=torch.float32).to(device)
            i_t = torch.tensor(X_te_slices[sm]).view(-1, 1, 128, 128).float().to(device)
            l_c_t = m_ct_fold(c_t, return_logit=True)
            t_un.append(torch.sigmoid(l_c_t + m_un_res(i_t)).mean().item())
        fold_preds_un.append(t_un)
    torch.cuda.empty_cache()

# Results
cv_probs["Attention U-Net"] = all_cv_probs_un
test_probs["Attention U-Net"] = np.mean(fold_preds_un, axis=0)
thresholds["Attention U-Net"] = np.mean(fold_ths_un)

Fold 1 Attention U-Net: 

25..50..75..100..Done.
Fold 2 Attention U-Net: 25..50..75..100..Done.
Fold 3 Attention U-Net: 25..50..75..100..Done.
Fold 4 Attention U-Net: 25..50..75..100..Done.
Fold 5 Attention U-Net: 25..50..75..100..Done.


In [3]:
# %%
from networks import LogisticRegression, MultiLayerPerceptron
import torch, torch.optim as optim
import torch.nn as nn
import numpy as np
from sklearn.preprocessing import StandardScaler


N_COMP = X_tr_clin.shape[1] 
LR_LR = 1e-4       
MLP_LR = 1e-4
MLP_HIDDEN_DIM = EMBED_DIM 

# Flush and initialize tracking arrays
all_cv_probs_lr, fold_preds_lr, fold_ths_lr = [], [], []
all_cv_probs_mlp, fold_preds_mlp, fold_ths_mlp = [], [], []

for fold, (t_subj_idx, v_subj_idx) in enumerate(eval_skf.split(unique_subjs, y_unique)):
    print(f"Fold {fold+1} LR & MLP: ", end='', flush=True)
    

    t_mask = np.isin(tr_subj_map, unique_subjs[t_subj_idx])
    scaler_c = StandardScaler().fit(X_tr_clin[t_mask])
    
    t_X_tr = torch.tensor(scaler_c.transform(X_tr_clin[t_mask]), dtype=torch.float32).to(device)
    t_y_tr = torch.tensor(y_tr_slices[t_mask], dtype=torch.float32).view(-1, 1).to(device)
    

    num_pos = t_y_tr.sum()
    num_neg = len(t_y_tr) - num_pos
    pos_w = (num_neg / (num_pos + 1e-5)).to(device)
    criterion_lr = nn.BCEWithLogitsLoss() 
    criterion_mlp = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    

    lr_model = LogisticRegression(N_COMP).to(device)
    lr_opt = optim.AdamW(lr_model.parameters(), lr=LR_LR)
    
    lr_model.train()
    for _ in range(EPOCHS): 
        lr_opt.zero_grad()
        logits = lr_model(t_X_tr)
        loss = criterion_lr(logits, t_y_tr)
        loss.backward()
        lr_opt.step()
        

    mlp_model = MultiLayerPerceptron(input_dim=N_COMP, hidden_dim=MLP_HIDDEN_DIM).to(device)
    mlp_opt = optim.AdamW(mlp_model.parameters(), lr=MLP_LR)
    
    mlp_model.train()
    for _ in range(EPOCHS): 
        mlp_opt.zero_grad()
        logits = mlp_model(t_X_tr)
        loss = criterion_mlp(logits, t_y_tr)
        loss.backward()
        mlp_opt.step()
        
    print('Training Done. Inferencing...', end='', flush=True)

    # Inference
    lr_model.eval()
    mlp_model.eval()
    with torch.no_grad():
        v_lr, v_mlp, v_lbls = [], [], []
        
        # Validation Inference
        for s_id in unique_subjs[v_subj_idx]:
            sm = (tr_subj_map == s_id)
            c_v = torch.tensor(scaler_c.transform(X_tr_clin[sm]), dtype=torch.float32).to(device)
            
            v_lr.append(torch.sigmoid(lr_model(c_v)).mean().item())
            v_mlp.append(torch.sigmoid(mlp_model(c_v)).mean().item())
            v_lbls.append(y_tr_slices[sm][0])
            

        fold_ths_lr.append(calibrate_balanced(PassThrough(), None, np.array(v_lr), np.array(v_lbls), 'cpu'))
        fold_ths_mlp.append(calibrate_balanced(PassThrough(), None, np.array(v_mlp), np.array(v_lbls), 'cpu'))
        
        all_cv_probs_lr.extend(v_lr)
        all_cv_probs_mlp.extend(v_mlp)

        # External Test Inference
        t_lr, t_mlp = [], []
        for s_id in np.unique(te_subj_map):
            sm = (te_subj_map == s_id)
            c_t = torch.tensor(scaler_c.transform(X_te_clin[sm]), dtype=torch.float32).to(device)
            
            t_lr.append(torch.sigmoid(lr_model(c_t)).mean().item())
            t_mlp.append(torch.sigmoid(mlp_model(c_t)).mean().item())
            
        fold_preds_lr.append(t_lr)
        fold_preds_mlp.append(t_mlp)
        
    print(' Done.')
    torch.cuda.empty_cache()

cv_probs["PCA+LR"] = all_cv_probs_lr
test_probs["PCA+LR"] = np.mean(fold_preds_lr, axis=0)
thresholds["PCA+LR"] = np.mean(fold_ths_lr)

cv_probs["PCA+MLP"] = all_cv_probs_mlp
test_probs["PCA+MLP"] = np.mean(fold_preds_mlp, axis=0)
thresholds["PCA+MLP"] = np.mean(fold_ths_mlp)

Fold 1 LR & MLP: 

Training Done. Inferencing... Done.
Fold 2 LR & MLP: Training Done. Inferencing... Done.
Fold 3 LR & MLP: Training Done. Inferencing... Done.
Fold 4 LR & MLP: Training Done. Inferencing... Done.
Fold 5 LR & MLP: Training Done. Inferencing... Done.


In [4]:
# Configuration
models = ["Clinical", "PCA+LR", "PCA+MLP", "Spectral", "Spatial", "Attention U-Net"]
n_bootstrap = 1000
n_permutations = 1000 
np.random.seed(0)

# Internal validation
print("\nInternal validation over 5 folds")
header_cv = f"{'Model':<16} | {'AUC':<18} | {'B-Acc':<18} | {'Spec':<18} | {'F1':<18}"
print("-" * len(header_cv))
print(header_cv)
print("-" * len(header_cv))

for name in models:
    if name not in cv_probs:
        continue
        
    m_mean = compute_comprehensive_metrics(
        np.array(all_cv_labels), np.array(cv_probs[name]), thresholds[name]
    )
    f_metrics = get_cv_fold_metrics(name, cv_probs, all_cv_labels, thresholds)

    n_folds = len(f_metrics)
    ci_mult = 2.776 / np.sqrt(n_folds) # t_{0.025, 4} = 2.776

    cells = {}
    for k in m_mean.keys():
        values = [f[k] for f in f_metrics]
        mean_val = np.mean(values)
        std_val = np.std(values, ddof=1) if n_folds > 1 else 0.0
        ci_half = ci_mult * (std_val / np.sqrt(n_folds)) if n_folds > 1 else 0.0
        cells[k] = (mean_val, std_val)

    print(
        f"{name:<16} | {fmt_cv('AUC',cells):<18} | {fmt_cv('B-Acc',cells):<18} |"
        f" {fmt_cv('Spec',cells):<18} | {fmt_cv('F1',cells):<18}"
    )


# External test
print("\n\n" + "=" * 200)
print("External test performance \nwith bootstrapped confidence intervals \nand paired permutation analysis")
print("=" * 200)

if "Spectral" in test_probs:
    y_true = np.array(y_test_labels)
    probs_spec = np.array(test_probs["Spectral"])
    thresh_spec = thresholds["Spectral"]
    n_samples = len(y_true)

    # Pre-calculate baseline 
    metrics_spec = compute_comprehensive_metrics(y_true, probs_spec, thresh_spec)

    # Expanded header
    header_ext = (
        f"{'Model':<16} | {'AUC':<20} | {'B-Acc':<20} | {'Spec':<20} | {'F1':<20} || "
        f"{'ΔAUC':<17} | {'ΔB-Acc':<17} | {'ΔSpec':<17} | {'ΔF1':<17}"
    )
    print(header_ext)
    print("-" * 200)

    for name in models:
        if name not in test_probs or name not in thresholds:
            continue

        probs_other = np.array(test_probs[name])
        thresh_other = thresholds[name]
        metrics_other = compute_comprehensive_metrics(y_true, probs_other, thresh_other)

        # Bootstrap for Confidence Intervals
        boot_results = {k: [] for k in ['AUC', 'B-Acc', 'Spec', 'F1']}
        for _ in range(n_bootstrap):
            idx = np.random.choice(n_samples, n_samples, replace=True)
            if len(np.unique(y_true[idx])) < 2: continue
            m_boot = compute_comprehensive_metrics(y_true[idx], probs_other[idx], thresh_other)
            for k in boot_results.keys():
                boot_results[k].append(m_boot[k])
        
        ci_display = {}
        for k in boot_results.keys():
            low, high = np.percentile(boot_results[k], [2.5, 97.5])
            ci_display[k] = f"{metrics_other[k]:.3f} [{low:.2f},{high:.2f}]"

        # Paired Permutation Test for all Metrics
        sig_cols = {}
        if name == "Spectral":
            sig_str = f" || {'-':<17} | {'-':<17} | {'-':<17} | {'-':<17}"
        else:
            for k in ['AUC', 'B-Acc', 'Spec', 'F1']:
                obs_diff = metrics_spec[k] - metrics_other[k] 
                perm_diffs = np.zeros(n_permutations)
                for i in range(n_permutations):
                    swap_mask = np.random.rand(n_samples) < 0.5
                    p_a = np.where(swap_mask, probs_spec, probs_other)
                    p_b = np.where(swap_mask, probs_other, probs_spec)
                    
                    m_a = compute_comprehensive_metrics(y_true, p_a, thresh_spec)
                    m_b = compute_comprehensive_metrics(y_true, p_b, thresh_other)
                    perm_diffs[i] = m_a[k] - m_b[k]

                # Directional p-value: probability of the Spectral lead being due to chance
                p_val = np.mean(perm_diffs >= obs_diff)
                
                sig = "*" if p_val < 0.05 else ""
                delta = metrics_other[k] - metrics_spec[k]
                sig_cols[k] = f"{delta:+.3f}{sig}"
            
            sig_str = (f" || {sig_cols['AUC']:<17} | {sig_cols['B-Acc']:<17} | "
                       f"{sig_cols['Spec']:<17} | {sig_cols['F1']:<17}")

        print(f"{name:<16} | {ci_display['AUC']:<20} | {ci_display['B-Acc']:<20} | "
              f"{ci_display['Spec']:<20} | {ci_display['F1']:<20}{sig_str}")

    print("-" * 200)
else:
    print("Error: 'Spectral' model results not found for comparison.")


Internal validation over 5 folds
----------------------------------------------------------------------------------------------------
Model            | AUC                | B-Acc              | Spec               | F1                
----------------------------------------------------------------------------------------------------
Clinical         | 0.810±0.089        | 0.745±0.078        | 0.794±0.129        | 0.812±0.028       
PCA+LR           | 0.558±0.146        | 0.521±0.126        | 0.603±0.340        | 0.587±0.120       
PCA+MLP          | 0.704±0.175        | 0.640±0.127        | 0.448±0.372        | 0.882±0.068       
Spectral         | 0.809±0.071        | 0.734±0.048        | 0.797±0.117        | 0.792±0.068       
Spatial          | 0.816±0.072        | 0.749±0.058        | 0.786±0.090        | 0.823±0.034       
Attention U-Net  | 0.877±0.041        | 0.773±0.064        | 0.814±0.070        | 0.834±0.076       


External test performance 
with bootstrapped confidence

In [13]:
# Configuration
models = ["Clinical", "PCA+LR", "PCA+MLP", "Spectral", "Spatial", "Attention U-Net"]
n_bootstrap = 1000
n_permutations = 1000 
block_size = 70  # Slices per patient for conservative grouped analysis
np.random.seed(0)

# Internal validation
print("\nInternal validation over 5 folds")
header_cv = f"{'Model':<16} | {'AUC':<18} | {'B-Acc':<18} | {'Spec':<18} | {'F1':<18}"
print("-" * len(header_cv))
print(header_cv)
print("-" * len(header_cv))

for name in models:
    if name not in cv_probs:
        continue
        
    m_mean = compute_comprehensive_metrics(
        np.array(all_cv_labels), np.array(cv_probs[name]), thresholds[name]
    )
    f_metrics = get_cv_fold_metrics(name, cv_probs, all_cv_labels, thresholds)

    n_folds = len(f_metrics)
    ci_mult = 2.776 / np.sqrt(n_folds) # t_{0.025, 4} = 2.776

    cells = {}
    for k in m_mean.keys():
        values = [f[k] for f in f_metrics]
        mean_val = np.mean(values)
        std_val = np.std(values, ddof=1) if n_folds > 1 else 0.0
        ci_half = ci_mult * (std_val / np.sqrt(n_folds)) if n_folds > 1 else 0.0
        cells[k] = (mean_val, std_val)

    print(
        f"{name:<16} | {fmt_cv('AUC',cells):<18} | {fmt_cv('B-Acc',cells):<18} |"
        f" {fmt_cv('Spec',cells):<18} | {fmt_cv('F1',cells):<18}"
    )


# External test
print("\n\n" + "=" * 200)
print("External test performance (Conservative Grouped Analysis)\n"
      f"Using Block Bootstrap/Permutation (Block Size ≈ {block_size} slices) to account for intra-subject correlation")
print("=" * 200)

if "Spectral" in test_probs:
    y_true = np.array(y_test_labels)
    probs_spec = np.array(test_probs["Spectral"])
    thresh_spec = thresholds["Spectral"]
    n_samples = len(y_true)
    n_blocks = int(np.ceil(n_samples / block_size))

    # Pre-calculate baseline Spectral Metrics
    metrics_spec = compute_comprehensive_metrics(y_true, probs_spec, thresh_spec)

    # Expanded Header
    header_ext = (
        f"{'Model':<16} | {'AUC [95% CI]':<20} | {'B-Acc [95% CI]':<20} | {'Spec [95% CI]':<20} | {'F1 [95% CI]':<20} || "
        f"{'ΔAUC':<17} | {'ΔB-Acc':<17} | {'ΔSpec':<17} | {'ΔF1':<17}"
    )
    print(header_ext)
    print("-" * 200)

    for name in models:
        if name not in test_probs or name not in thresholds:
            continue

        probs_other = np.array(test_probs[name])
        thresh_other = thresholds[name]
        metrics_other = compute_comprehensive_metrics(y_true, probs_other, thresh_other)

        # --- BLOCK BOOTSTRAP FOR CI ---
        boot_results = {k: [] for k in ['AUC', 'B-Acc', 'Spec', 'F1']}
        for _ in range(n_bootstrap):
            # Sample blocks (patients) instead of individual slices
            block_idx = np.random.choice(n_blocks, n_blocks, replace=True)
            indices = []
            for b in block_idx:
                start = b * block_size
                end = min((b + 1) * block_size, n_samples)
                indices.extend(np.arange(start, end))
            indices = np.array(indices)
            
            if len(np.unique(y_true[indices])) < 2: continue
            m_boot = compute_comprehensive_metrics(y_true[indices], probs_other[indices], thresh_other)
            for k in boot_results.keys():
                boot_results[k].append(m_boot[k])
        
        ci_display = {}
        for k in boot_results.keys():
            low, high = np.percentile(boot_results[k], [2.5, 97.5])
            ci_display[k] = f"{metrics_other[k]:.3f} [{low:.2f},{high:.2f}]"

        # --- PAIRED BLOCK PERMUTATION TEST ---
        sig_cols = {}
        if name == "Spectral":
            sig_str = f" || {'-':<17} | {'-':<17} | {'-':<17} | {'-':<17}"
        else:
            for k in ['AUC', 'B-Acc', 'Spec', 'F1']:
                obs_diff = metrics_spec[k] - metrics_other[k] 
                perm_diffs = np.zeros(n_permutations)
                
                for i in range(n_permutations):
                    # One coin flip per block (subject)
                    block_swap = np.random.rand(n_blocks) < 0.5
                    # Expand mask to all slices in block
                    swap_mask = np.repeat(block_swap, block_size)[:n_samples]
                    
                    p_a = np.where(swap_mask, probs_other, probs_spec)
                    p_b = np.where(swap_mask, probs_spec, probs_other)
                    
                    m_a = compute_comprehensive_metrics(y_true, p_a, thresh_spec)
                    m_b = compute_comprehensive_metrics(y_true, p_b, thresh_other)
                    perm_diffs[i] = m_a[k] - m_b[k]

                # Directional p-value: prob of the lead being due to chance
                # Using >= because obs_diff is (Spectral - Other)
                p_val = np.mean(perm_diffs >= obs_diff)
                
                sig = "*" if p_val < 0.05 else ""
                delta = metrics_other[k] - metrics_spec[k]
                sig_cols[k] = f"{delta:+.3f}{sig}"
            
            sig_str = (f" || {sig_cols['AUC']:<17} | {sig_cols['B-Acc']:<17} | "
                       f"{sig_cols['Spec']:<17} | {sig_cols['F1']:<17}")

        print(f"{name:<16} | {ci_display['AUC']:<20} | {ci_display['B-Acc']:<20} | "
              f"{ci_display['Spec']:<20} | {ci_display['F1']:<20}{sig_str}")

    print("-" * 200)
else:
    print("Error: 'Spectral' model results not found for comparison.")


Internal validation over 5 folds
----------------------------------------------------------------------------------------------------
Model            | AUC                | B-Acc              | Spec               | F1                
----------------------------------------------------------------------------------------------------
Clinical         | 0.810±0.089        | 0.745±0.078        | 0.794±0.129        | 0.812±0.028       
PCA+LR           | 0.590±0.093        | 0.530±0.068        | 0.363±0.262        | 0.785±0.129       
PCA+MLP          | 0.610±0.057        | 0.539±0.055        | 0.580±0.386        | 0.603±0.262       
Spectral         | 0.809±0.071        | 0.734±0.048        | 0.797±0.117        | 0.792±0.068       
Spatial          | 0.816±0.072        | 0.749±0.058        | 0.786±0.090        | 0.823±0.034       
Attention U-Net  | 0.877±0.041        | 0.773±0.064        | 0.814±0.070        | 0.834±0.076       


External test performance (Conservative Grouped Analysi